# Exercise 3: Elliptic Curve Cryptography (ECDSA)

In [ ]:
# Install if needed: pip install ecdsa
from ecdsa import SigningKey, SECP256k1, VerifyingKey
from ecdsa.util import sigencode_der, sigdecode_der
import hashlib

## Step 1: Generate ECDSA key pair (SECP256k1 - same curve used by Bitcoin/Ethereum)

In [ ]:
private_key = SigningKey.generate(curve=SECP256k1)
public_key = private_key.get_verifying_key()

# Convert to hex format
private_key_hex = private_key.to_string().hex()
public_key_hex = public_key.to_string("uncompressed").hex()

print("Private Key (hex):")
print(private_key_hex)
print()
print("Public Key - Uncompressed (hex):")
print(public_key_hex)

## Step 2: Compare key sizes (bytes and hex characters)

In [ ]:
private_key_bytes = private_key.to_string()
public_key_bytes = public_key.to_string("uncompressed")

print(f"Private Key size: {len(private_key_bytes)} bytes, {len(private_key_hex)} hex characters")
print(f"Public Key (uncompressed) size: {len(public_key_bytes)} bytes, {len(public_key_hex)} hex characters")

## Step 3: Compress the public key

An uncompressed public key stores both the X and Y coordinates of the point on the curve (prefix `04`).
A compressed public key stores only the X coordinate plus a 1-byte prefix (`02` or `03`) indicating whether Y is even or odd -- since Y can be mathematically recovered from X and the curve equation.

In [ ]:
def compress_public_key(vk: VerifyingKey) -> bytes:
    """Compress an EC public key: prefix (02/03) + X coordinate."""
    point = vk.pubkey.point
    x = point.x()
    y = point.y()

    prefix = b"\x02" if y % 2 == 0 else b"\x03"
    x_bytes = x.to_bytes(32, byteorder="big")

    return prefix + x_bytes


compressed_public_key = compress_public_key(public_key)
compressed_public_key_hex = compressed_public_key.hex()

print("Public Key - Compressed (hex):")
print(compressed_public_key_hex)
print()
print(f"Compressed Public Key size: {len(compressed_public_key)} bytes, {len(compressed_public_key_hex)} hex characters")

reduction_pct = (1 - len(compressed_public_key) / len(public_key_bytes)) * 100
print(f"\nStorage reduction: {reduction_pct:.2f}%")

## Step 4: Decompress the public key (to verify it can be reconstructed)

In [ ]:
def decompress_public_key(compressed: bytes, curve=SECP256k1) -> VerifyingKey:
    """Decompress an EC public key back into a full (X, Y) point."""
    prefix = compressed[0]
    x = int.from_bytes(compressed[1:], byteorder="big")

    curve_obj = curve.curve
    p = curve_obj.p()
    a = curve_obj.a()
    b = curve_obj.b()

    # y^2 = x^3 + a*x + b (mod p)
    y_squared = (x**3 + a * x + b) % p
    y = pow(y_squared, (p + 1) // 4, p)  # modular sqrt (p % 4 == 3 for secp256k1)

    # Choose the correct root based on the prefix (02 = even, 03 = odd)
    if (y % 2 == 0 and prefix == 0x03) or (y % 2 == 1 and prefix == 0x02):
        y = p - y

    from ecdsa.ellipticcurve import Point
    point = Point(curve_obj, x, y)
    return VerifyingKey.from_public_point(point, curve=curve)


decompressed_key = decompress_public_key(compressed_public_key)
decompressed_hex = decompressed_key.to_string("uncompressed").hex()

print("Decompressed Public Key (hex):")
print(decompressed_hex)
print()
print("Matches original uncompressed key?", decompressed_hex == public_key_hex)

## Step 5: Sign a message and verify using the decompressed public key

In [ ]:
message = b"This is a test message for COMP842 Exercise 3."

# Sign the message using the private key
signature = private_key.sign(message, hashfunc=hashlib.sha256, sigencode=sigencode_der)
print("Signature (hex):")
print(signature.hex())

In [ ]:
# Verify using the ORIGINAL (uncompressed) public key
is_valid_original = public_key.verify(signature, message, hashfunc=hashlib.sha256, sigdecode=sigdecode_der)
print("Signature valid using original public key:", is_valid_original)

# Verify using the DECOMPRESSED public key (rebuilt from the compressed form)
is_valid_decompressed = decompressed_key.verify(signature, message, hashfunc=hashlib.sha256, sigdecode=sigdecode_der)
print("Signature valid using decompressed public key:", is_valid_decompressed)

## Summary

This confirms that:
- The compressed public key stores only 33 bytes vs 65 bytes for the uncompressed key.
- The compressed key can be fully decompressed back into the original (X, Y) point.
- The decompressed key successfully verifies a digital signature, proving no cryptographic information was lost by compressing it.